# Modeling and Deployment

## 1) Importing necessary libraries

In [1]:
import joblib
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
import warnings
warnings.filterwarnings('ignore')
print("Imports done!")

Imports done!


## Load Data + Index-Based Split

In [2]:
train_data = joblib.load('encoded_train_data.joblib', mmap_mode='r')
test_data  = joblib.load('encoded_test_data.joblib',  mmap_mode='r')
 
feature_cols = [col for col in train_data.columns if col != 'IncidentGrade']
y_all = np.array(train_data['IncidentGrade'])  # sirf 1 col copy — safe hai
 
# Index-based 80:20 split (no data copy hoga)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(y_all)), y_all))
print(f"Train indices: {len(train_idx)} | Val indices: {len(val_idx)}")

Train indices: 7138244 | Val indices: 1784561


## Helper Function: Load Sample

In [3]:
def load_sample(idx, sample_frac=0.1, convert_bool=True):
    """
    idx         : train_idx ya val_idx
    sample_frac : kitna % sample karna hai (0.1 = 10%)
    """
    n = int(len(idx) * sample_frac)
    sampled = np.random.choice(idx, size=n, replace=False)
    X = train_data[feature_cols].iloc[sampled].values.astype(np.float32)
    y = y_all[sampled]
    return X, y
 
print("Helper function ready!")

Helper function ready!


# 2) Comparing Machine Learning Models

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
 
print("Loading 10% train sample for comparison...")
X_train_s, y_train_s = load_sample(train_idx, sample_frac=0.1)
 
# Validation sample (100k rows enough hai evaluate karne ke liye)
val_sample = val_idx[:100_000]
X_val_s = train_data[feature_cols].iloc[val_sample].values.astype(np.float32)
y_val_s = y_all[val_sample]
 
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_jobs=-1, random_state=42),
    'XGBoost':             XGBClassifier(n_jobs=-1, random_state=42, verbosity=0),
    'LightGBM':            LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1),
}
 
results = []
for name, model in models.items():
    print(f"\nTraining: {name} ...")
    model.fit(X_train_s, y_train_s)
    preds = model.predict(X_val_s)
    acc = accuracy_score(y_val_s, preds)
    report = classification_report(y_val_s, preds, output_dict=True)
    macro_f1 = report['macro avg']['f1-score']
    results.append({'Model': name, 'Accuracy': round(acc, 4),
                    'Macro-F1': round(macro_f1, 2)})
    print(f"  Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.2f}")
 
df_results = pd.DataFrame(results)
print("\n=== Comparison Table ===")
print(df_results.to_string(index=False))
 
best = df_results.loc[df_results['Macro-F1'].idxmax()]
print(f"\nBest Model: {best['Model']} (Macro-F1: {best['Macro-F1']})")

Loading 10% train sample for comparison...

Training: Logistic Regression ...
  Accuracy: 0.6319 | Macro-F1: 0.54

Training: Decision Tree ...
  Accuracy: 0.7032 | Macro-F1: 0.67

Training: Random Forest ...
  Accuracy: 0.7045 | Macro-F1: 0.67

Training: XGBoost ...
  Accuracy: 0.6794 | Macro-F1: 0.62

Training: LightGBM ...
  Accuracy: 0.6773 | Macro-F1: 0.61

=== Comparison Table ===
              Model  Accuracy  Macro-F1
Logistic Regression    0.6319      0.54
      Decision Tree    0.7032      0.67
      Random Forest    0.7045      0.67
            XGBoost    0.6794      0.62
           LightGBM    0.6773      0.61

Best Model: Decision Tree (Macro-F1: 0.67)


## Random Forest WITH SMOTE (2% sample)

In [5]:
print("Loading 2% sample for SMOTE training...")
X_train_sm, y_train_sm = load_sample(train_idx, sample_frac=0.02)
 
# Bool columns ko int mein convert karo (SMOTE ke liye)
# Already float32 hai toh direct use karo
 
print("Applying SMOTE...")
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_sm, y_train_sm)
print(f"After SMOTE: {X_resampled.shape}")
 
param_dist = {
    'n_estimators':     [50, 75],
    'max_depth':        [10, 20, None],
    'min_samples_split':[2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap':        [True, False]
}
 
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
random_search = RandomizedSearchCV(rf, param_dist, n_iter=5,
                                   cv=3, verbose=1, random_state=42, n_jobs=-1)
print("Training RF with SMOTE...")
random_search.fit(X_resampled, y_resampled)
best_rf_smote = random_search.best_estimator_
 
# Validate
X_val_s = train_data[feature_cols].iloc[val_idx[:100_000]].values.astype(np.float32)
y_val_s  = y_all[val_idx[:100_000]]
y_pred = best_rf_smote.predict(X_val_s)
 
print("\nBest Params:", random_search.best_params_)
print("\nClassification Report (RF + SMOTE):")
print(classification_report(y_val_s, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_val_s, y_pred))
 
joblib.dump(best_rf_smote, "rf_smote_tuned_model.joblib")
print("Saved: rf_smote_tuned_model.joblib")

Loading 2% sample for SMOTE training...
Applying SMOTE...
After SMOTE: (184926, 153)
Training RF with SMOTE...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Best Params: {'n_estimators': 75, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None, 'bootstrap': False}

Classification Report (RF + SMOTE):
              precision    recall  f1-score   support

           0       0.68      0.75      0.71     42530
           1       0.53      0.52      0.52     21813
           2       0.74      0.67      0.71     35657

    accuracy                           0.67    100000
   macro avg       0.65      0.65      0.65    100000
weighted avg       0.67      0.67      0.67    100000

Confusion Matrix:
[[31769  6129  4632]
 [ 6808 11269  3736]
 [ 7842  3823 23992]]
Saved: rf_smote_tuned_model.joblib


## Random Forest WITHOUT SMOTE (2% sample)

In [6]:
print("Loading 2% sample (no SMOTE)...")
X_train_ns, y_train_ns = load_sample(train_idx, sample_frac=0.02)
 
rf_no_smote = RandomForestClassifier(random_state=42, n_jobs=-1)
print("Training RF without SMOTE...")
rf_no_smote.fit(X_train_ns, y_train_ns)
 
X_val_s = train_data[feature_cols].iloc[val_idx[:100_000]].values.astype(np.float32)
y_val_s  = y_all[val_idx[:100_000]]
y_pred_ns = rf_no_smote.predict(X_val_s)
 
print("\nClassification Report (RF, No SMOTE):")
print(classification_report(y_val_s, y_pred_ns))
print("Confusion Matrix:")
print(confusion_matrix(y_val_s, y_pred_ns))
 
joblib.dump(rf_no_smote, "rf_no_smote_model.joblib")
print("Saved: rf_no_smote_model.joblib")

Loading 2% sample (no SMOTE)...
Training RF without SMOTE...

Classification Report (RF, No SMOTE):
              precision    recall  f1-score   support

           0       0.67      0.80      0.73     42530
           1       0.62      0.43      0.51     21813
           2       0.73      0.70      0.71     35657

    accuracy                           0.68    100000
   macro avg       0.67      0.64      0.65    100000
weighted avg       0.68      0.68      0.67    100000

Confusion Matrix:
[[33947  3482  5101]
 [ 8198  9382  4233]
 [ 8385  2389 24883]]
Saved: rf_no_smote_model.joblib


## Random Forest WITH SMOTEENN (2% sample)

In [7]:
print("Loading 2% sample for SMOTEENN training...")
X_train_se, y_train_se = load_sample(train_idx, sample_frac=0.02)
 
print("Applying SMOTEENN (takes a bit longer)...")
smote_enn = SMOTEENN(random_state=42)
X_res_enn, y_res_enn = smote_enn.fit_resample(X_train_se, y_train_se)
print(f"After SMOTEENN: {X_res_enn.shape}")
 
rf2 = RandomForestClassifier(random_state=42, n_jobs=-1)
rs2 = RandomizedSearchCV(rf2, param_dist, n_iter=5,
                          cv=3, verbose=1, random_state=42, n_jobs=-1)
print("Training RF with SMOTEENN...")
rs2.fit(X_res_enn, y_res_enn)
best_rf_enn = rs2.best_estimator_
 
X_val_s = train_data[feature_cols].iloc[val_idx[:100_000]].values.astype(np.float32)
y_val_s  = y_all[val_idx[:100_000]]
y_pred_enn = best_rf_enn.predict(X_val_s)
 
print("\nBest Params:", rs2.best_params_)
print("\nClassification Report (RF + SMOTEENN):")
print(classification_report(y_val_s, y_pred_enn))
print("Confusion Matrix:")
print(confusion_matrix(y_val_s, y_pred_enn))
 
joblib.dump(best_rf_enn, "rf_smote_enn_tuned_model.joblib")
print("Saved: rf_smote_enn_tuned_model.joblib")

Loading 2% sample for SMOTEENN training...
Applying SMOTEENN (takes a bit longer)...
After SMOTEENN: (67912, 153)
Training RF with SMOTEENN...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Best Params: {'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'bootstrap': False}

Classification Report (RF + SMOTEENN):
              precision    recall  f1-score   support

           0       0.66      0.74      0.70     42530
           1       0.49      0.45      0.47     21813
           2       0.73      0.67      0.70     35657

    accuracy                           0.65    100000
   macro avg       0.63      0.62      0.62    100000
weighted avg       0.65      0.65      0.65    100000

Confusion Matrix:
[[31307  6365  4858]
 [ 7988  9732  4093]
 [ 8147  3676 23834]]
Saved: rf_smote_enn_tuned_model.joblib


## Evaluate All 3 Models on TEST DATA

In [8]:
print("Loading test data...")
X_test = test_data[feature_cols].values.astype(np.float32)
y_test = np.array(test_data['IncidentGrade'])
 
for model_file, label in [
    ("rf_smote_tuned_model.joblib",     "RF + SMOTE"),
    ("rf_no_smote_model.joblib",        "RF No SMOTE"),
    ("rf_smote_enn_tuned_model.joblib", "RF + SMOTEENN"),
]:
    print(f"\n{'='*50}")
    print(f"Model: {label}")
    m = joblib.load(model_file)
    y_pred_test = m.predict(X_test)
    report = classification_report(y_test, y_pred_test, output_dict=True)
    print(classification_report(y_test, y_pred_test))
    print(f"Macro-F1    : {report['macro avg']['f1-score']:.2f}")
    print(f"Macro Prec  : {report['macro avg']['precision']:.2f}")
    print(f"Macro Recall: {report['macro avg']['recall']:.2f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_test))

Loading test data...

Model: RF + SMOTE
              precision    recall  f1-score   support

           0       0.64      0.71      0.67   1630942
           1       0.46      0.44      0.45    868897
           2       0.71      0.64      0.67   1422856

    accuracy                           0.62   3922695
   macro avg       0.60      0.60      0.60   3922695
weighted avg       0.62      0.62      0.62   3922695

Macro-F1    : 0.60
Macro Prec  : 0.60
Macro Recall: 0.60
Confusion Matrix:
[[1158980  266030  205932]
 [ 326193  378303  164401]
 [ 327355  185286  910215]]

Model: RF No SMOTE
              precision    recall  f1-score   support

           0       0.64      0.76      0.69   1630942
           1       0.53      0.38      0.44    868897
           2       0.70      0.66      0.68   1422856

    accuracy                           0.64   3922695
   macro avg       0.62      0.60      0.61   3922695
weighted avg       0.64      0.64      0.63   3922695

Macro-F1    : 0.61
Ma

In [10]:
import joblib

train_data = joblib.load('encoded_train_data.joblib', mmap_mode='r')
feature_cols = [col for col in train_data.columns if col != 'IncidentGrade']

# E drive path — jahan models already hain
joblib.dump(feature_cols, r'E:\AI & ML Development\Machine Learning\Projects\Cyber Security Threat Classifier\models\feature_cols.joblib')
print(f"Saved! Total features: {len(feature_cols)}")

Saved! Total features: 153
